### DESCRIPTION (inspired by Leetcode.com)
A hiking app needs to find the easiest route across terrain. Given a 2D grid where each cell represents the elevation at that point, find the path from the top-left corner to the bottom-right corner that minimizes the "effort".

The effort of moving between two adjacent cells is the absolute difference in their elevations. The effort of a path is its single hardest step, meaning the largest effort among all the moves you make along the way.

You can move up, down, left, or right from any cell. Return the minimum effort required.

Example 1:

Input:

heights = [[1,10,2], [2,3,3], [3,2,1]]

Output: 1

Explanation: The cliff (10) has a huge elevation difference. Going around via 1→2→3→2→1 avoids it entirely, with each step having effort at most 1.

Example 2:

Input:
heights = [[4,3,2], [2,6,3], [3,2,1]]

Output:2

Explanation: The center peak (6) would require effort 3 to cross. Going around the top: 4→3→2→3→1 keeps maximum effort at 2.

In [14]:
from typing import List
import heapq

class Solution:
    def minimumEffortPath(self, heights: List[List[int]]) -> int:
        if not heights or not heights[0]:
            return -1
        
        largest_effort = 0
        directions = [(0,-1),(-1,0),(0,1),(1,0)]
        rows, cols = len(heights), len(heights[0])

        # 1. starting from left top cell, put its (effort, r, c) to a min-heap and visited set 
        heap = [(0, 0, 0)]
        visited = {(0, 0, 0)}

        # 2. while heap is not empty, pop an item. Compare against largest_effort, update if effort is larger. 
        while heap:
            effort, r, c = heapq.heappop(heap)
            largest_effort = max(largest_effort, effort)

            # 3. If r,c is the bottome right cell, return largest_effort.
            if r == rows-1 and c == cols-1:
                return largest_effort
            
            # 4. If not, check 4 directions of r,c. If nr, nc are within bounds, calculate new_effort between r,c and nr, nc.
            # # If (new_effort, nr, nc) not in visited set, add to heap and visited set.       
            for dr, dc in directions:
                nr = r+dr
                nc = c+dc
                if nr >= 0 and nr < rows and nc >= 0 and nc < cols:
                    new_effort = abs(heights[r][c]-heights[nr][nc])
                    if (new_effort, nr, nc) not in visited:
                        heapq.heappush(heap, (new_effort, nr, nc))
                        visited.add((new_effort, nr, nc))

### Feedback

Your solution passes the tests and correctly uses a min-heap to explore routes. However, the heap key is only the most recent edge difference, while path effort is the maximum difference across the entire path. largest_effort is global, so it may include an edge from an unrelated explored route; this makes the logic fragile and can return an incorrect result on other inputs. Track each candidate’s accumulated effort instead: path_effort = max(effort, abs(...)), and push (path_effort, nr, nc). Also, visited should be per cell (or use a distance matrix), because reaching a cell with a smaller accumulated effort should be allowed even if it was previously seen. A standard distance matrix gives correctness in O(RC log(RC)) time and O(RC) space.

In [63]:
from typing import List
import heapq

class Solution:
    def minimumEffortPath(self, heights: List[List[int]]) -> int:
        if not heights or not heights[0]:
            return -1
        
        directions = [(0,-1),(-1,0),(0,1),(1,0)]
        rows, cols = len(heights), len(heights[0])

        # 1. starting from left top cell, put (effort, r, c) to a min-heap. Create a distance matrix to track largest effort to each cell. 
        heap = [(0, 0, 0)]
        dist = [[float('inf')] * cols for _ in range(rows)]
        dist[0][0] = 0

        # 2. while heap is not empty, pop an item. Return effort if cell is bottom right or skip if effort is larger than what we have in distance matrix.  
        while heap:
            path_effort, r, c = heapq.heappop(heap)
            
            if r == rows-1 and c == cols-1:
                return path_effort

            if path_effort > dist[r][c]:
                continue

            # If not, check 4 directions of r,c. If nr, nc are within bounds, calculate new_effort between r,c and nr, nc.
            # # If (new_effort, nr, nc) is less than that in distance matrix, add to heap and update matrix.       
            for dr, dc in directions:
                nr = r+dr
                nc = c+dc
                if 0 <= nr < rows and 0 <= nc < cols:
                    diff = abs(heights[nr][nc]-heights[r][c])
                    new_effort = max(path_effort, diff)

                    if new_effort < dist[nr][nc]:
                        dist[nr][nc] = new_effort
                        heapq.heappush(heap, (new_effort, nr, nc))
                                                
        return dist[rows-1][cols-1]

In [44]:
from typing import Callable

class Test:  
    def __init__(self, input: List[List[int]], result: int):
        self.input = input
        self.expected_result = result
        
def run_tests(tests: list[Test], func: Callable[[List[List[int]]], int]):
    for test in tests:
        result = func(test.input)
        if result == test.expected_result:
            print(f"Test passed for {test.input}")
        else:
            print(f"Test failed for {test.input}. Expected: {test.expected_result}, Actual: {result}")

In [64]:
tests = [
    Test(([[]]), -1),
    Test(([[2]]), 0),
    Test(([[1,2]]), 1),
    Test(([[1],[2]]), 1),
    Test(([[1,10,2], [2,3,3], [3,2,1]]), 1),
    Test(([[4,3,2], [2,6,3], [3,2,1]]), 2)
]

run_tests(tests, Solution().minimumEffortPath)

Test passed for [[]]
Test passed for [[2]]
Test passed for [[1, 2]]
Test passed for [[1], [2]]
Test passed for [[1, 10, 2], [2, 3, 3], [3, 2, 1]]
Test passed for [[4, 3, 2], [2, 6, 3], [3, 2, 1]]
